In [ ]:
# Imports and configuration
import warnings
warnings.filterwarnings('ignore')
import re, time, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
                             davies_bouldin_score, adjusted_rand_score)
from statsmodels.tsa.stattools import coint
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
from pykalman import KalmanFilter

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

# Data
DATA_START   = '2000-01-01'   # one buffer year so the 52-week rolling Sharpe covers all of Train
END_DATE     = '2026-01-01'
SNAPSHOT     = '2001-01-01'   # constituent snapshot date
BENCHMARK    = 'SP500TR'          # auto-adjusted, so dividends are included
RF_TICKER    = '^IRX'         # 13-week T-bill yield, annualized percent
FALLBACK_RF  = 0.0            # used only if ^IRX fails to download

# Periods
TRAIN_START, TRAIN_END = '2001-01-01', '2008-12-31'   # 8 years
VAL_START,   VAL_END   = '2009-01-01', '2016-12-31'   # 8 years
TEST_START,  TEST_END  = '2017-01-01', '2025-12-31'   # 9 years

# Clustering
ROLLING_WINDOW = 52
K_VALUES       = [2, 4, 7]
K_COINT        = 7      # pair formation uses k=7 clusters to keep the candidate pool tractable
N_CORES        = 10     # only used by the multiprocess fallback for OCP

# Strategy parameters 
ZSCORE_WINDOW  = 52
VOL_WINDOW     = 26
VOL_THRESHOLD  = 1.5
HIGH_VOL_ENTRY = 2.25
LOW_VOL_ENTRY  = 1.25
STANDARD_ENTRY = 1.75
EXIT_THRESHOLD = 0.5
STOP_LOSS      = 3.25

CANDIDATE_MULTIPLIERS = [0.60, 0.75, 0.85, 1.00, 1.15, 1.25]
ACTIVITY_FLOOR        = 0.50
COST_LEVELS_BPS       = [0, 5, 10, 20]

N_BOOT, BLOCK_SIZE, BOOT_SEED = 5000, 8, 42

# calendar version (robustness). Everything outside the windows is bull.
BEAR_WINDOWS_PRIMARY = [('2018-09-20', '2018-12-24'),
                        ('2020-02-19', '2020-03-23'),
                        ('2022-01-03', '2022-10-12')]
BEAR_WINDOWS_ALT     = [('2018-10-01', '2018-12-31'),
                        ('2020-02-19', '2020-03-23'),
                        ('2022-01-03', '2022-10-12')]

METHODS = ['DTW', 'OCP', 'CORR']
LABELS  = {'DTW': 'DTW-Clustering', 'OCP': 'OCP-Clustering',
           'CORR': 'Correlation Pairs', 'Naive': 'Naive Buy-and-Hold'}
COLORS  = {'DTW': '#1f77b4', 'OCP': '#ff7f0e', 'CORR': '#2ca02c', 'Naive': '#7f7f7f'}
TAG     = 'sp500_25y'

def n_years(a, b):
    return round((pd.Timestamp(b) - pd.Timestamp(a)).days / 365.25)

print('DTW vs OCP vs Correlation Pairs vs Naive, S&P 500 25-Year Framework')
print(f'Train: {TRAIN_START} to {TRAIN_END} ({n_years(TRAIN_START, TRAIN_END)} years)')
print(f'Val:   {VAL_START} to {VAL_END} ({n_years(VAL_START, VAL_END)} years)')
print(f'Test:  {TEST_START} to {TEST_END} ({n_years(TEST_START, TEST_END)} years)')
